# Stage 4: Kaggle experiment readiness

This notebook loads **saved software-validation aggregates only**. Run All makes no API calls and runs no primary-window policy episodes. Stage 4 verifies the experiment interface and execution safeguards; Stage 5 will generate primary comparisons. Scripted fixtures are not genuine LLM performance evidence.

In [1]:
import hashlib, json
from pathlib import Path
import pandas as pd
from IPython.display import display
ROOT=Path.cwd()
if not (ROOT/'docs/results/stage4').exists():
    ROOT=ROOT.parent
PUBLIC=ROOT/'docs/results/stage4'
manifest=json.loads((PUBLIC/'artifact_manifest.json').read_text())
for name,digest in manifest['files'].items():
    assert hashlib.sha256((PUBLIC/name).read_bytes().replace(b'\r\n',b'\n')).hexdigest()==digest,name
status=json.loads((PUBLIC/'status.json').read_text())
qc=json.loads((PUBLIC/'qc_summary.json').read_text())
assert qc['exit_gate_passed'] and all(qc['checks'].values())
assert qc['new_model_completion_calls']==qc['new_primary_policy_episodes']==0
display(pd.Series(status))
display(pd.Series({k:qc[k] for k in ['automated_tests','execution_and_replay_regression_tests','frozen_design_checks']}))

stage                                                                                      4
status                                                            complete_offline_readiness
exit_gate_passed                                                                        True
primary_comparisons_executed                                                           False
paid_primary_authorized                                                                False
next_stage                                                                                 5
next_stage_work                            Explicitly authorized matched primary baseline...
native_coverage_qualification              14 validated non-Retail tasks; frozen ICBE fai...
primary_window_prior_manual_outcome_use             uncertain; no pristine unseen-data claim
dtype: object

automated_tests                          82
execution_and_replay_regression_tests    16
frozen_design_checks                     31
dtype: int64

## Expanded validation matrix

Three training-selected series, four scenarios, and fixed/adaptive/scripted policies produce 36 validation episodes and 756 policy-days. All use 2?22 April 2023. The scripted client deliberately generates a repaired decision and an adaptive fallback to verify their export; it never contacts a model.

The independent validator reconstructs frozen inputs and environment transitions. It checks past-only histories, hidden private demand on every day, scenario privacy except legitimate current lead time, matching initial conditions and base-demand paths, policy actions, and monetary/inventory/pipeline conservation.

In [2]:
coverage=pd.read_csv(PUBLIC/'validation_coverage.csv')
assert len(coverage)==36 and coverage.days.sum()==756
assert coverage.status.eq('validated_complete').all() and coverage.model_completion_calls.eq(0).all()
assert coverage.loc[coverage.agent_kind=='baseline','legacy_baseline_parity'].all()
display(coverage.groupby(['store_id','product_id','scenario','agent_kind']).size().unstack(fill_value=0).rename_axis(columns='Validation episode kind'))
display(pd.Series(qc['checks']))
scripted=coverage.loc[coverage.agent_kind=='scripted']
assert len(scripted)==12 and scripted.scripted_repair_days.eq(2).all() and scripted.scripted_fallback_days.eq(1).all()
print('Scripted-only fixture events:', {'repair_days':int(scripted.scripted_repair_days.sum()),'fallback_days':int(scripted.scripted_fallback_days.sum())})

Validation episode kind                  baseline  scripted
store_id product_id scenario                               
S001     P0001      control                     2         1
                    delivery_disruption         2         1
                    demand_decline              2         1
                    demand_surge                2         1
S003     P0016      control                     2         1
                    delivery_disruption         2         1
                    demand_decline              2         1
                    demand_surge                2         1
S004     P0006      control                     2         1
                    delivery_disruption         2         1
                    demand_decline              2         1
                    demand_surge                2         1

expanded_validation_matrix                                      True
all_36_independent_replays                                      True
all_24_legacy_baselines_match                                   True
matched_initial_conditions_and_demand                           True
daily_future_demand_and_scenario_privacy                        True
cash_inventory_pipeline_reconciled                              True
scripted_repairs_fallbacks_preserved                            True
two_genuine_pilots_revalidated_without_transport                True
authorization_duplicate_retry_lock_and_attempt_ceiling_tests    True
prior_ledger_spending_and_holds_unchanged                       True
frozen_design_unchanged                                         True
no_primary_evaluation_or_new_model_calls                        True
dtype: bool

Scripted-only fixture events: {'repair_days': 24, 'fallback_days': 12}


## Genuine validation evidence retained from Stage 2

Two earlier genuine 21-day LLM pilots were revalidated using their actual saved requests/responses, without reissuing their 57 requests. Their 42 decisions included 15 repairs and three adaptive fallbacks. The LLM operational policy underperformed the adaptive comparator in both validation episodes. Prompts, schema, model and fallback policy remain frozen; these are not primary-window findings.

In [3]:
pilots=pd.DataFrame(json.loads((ROOT/'docs/results/stage2/kaggle_validation_summary.json').read_text()))
llm=pilots.loc[pilots.policy=='llm_with_fallback']
display(pilots[['scenario','policy','days','contribution','fill_rate','repair_days','fallback_days']])
opportunities=int(llm.decision_opportunities.sum())
repairs=int(llm.repair_days.sum())
fallbacks=int(llm.fallback_days.sum())
print({'historical_decisions':opportunities,'first_pass_valid':opportunities-repairs,'model_completion_after_repair':opportunities-fallbacks,'fallback_days':fallbacks,'new_model_calls':0})
assert qc['revalidated_historical_genuine_pilots']==2 and qc['historical_attempts_audited_not_reissued']==57

,scenario,policy,days,contribution,fill_rate,repair_days,fallback_days
0,control,llm_with_fallback,21,35313.166875,0.971304,7,1
1,delivery_disruption,llm_with_fallback,21,35854.402500,0.981784,8,2
2,control,fixed,21,38527.033125,0.980718,0,0
3,control,adaptive,21,40167.986250,0.998621,0,0
4,delivery_disruption,fixed,21,37715.829375,0.958777,0,0
5,delivery_disruption,adaptive,21,39436.451250,0.975610,0,0


{'historical_decisions': 42, 'first_pass_valid': 27, 'model_completion_after_repair': 39, 'fallback_days': 3, 'new_model_calls': 0}


## Primary comparisons remain planned

The unchanged primary matrix has three selected series ? four scenarios ? three policies: 36 episodes, including 12 LLM episodes and 252 first attempts (at most 504 attempts with repairs). The 22 September?12 October 2023 window has **uncertain prior manual outcome use**; no pristine unseen-data claim is allowed. Three selected matched blocks support descriptive comparisons, not inference treating daily rows as independent replications.

Execution preserves canonical episode identities and an immutable matrix registry. One episode runs at a time. Complete/active duplicates are rejected; interrupted episodes require an evidence-bound review and a fresh environment. Paid episodes require validated matching baselines and cumulative authorization. Fallback above 5% requires review before another paid episode.

In [4]:
plan=pd.read_csv(PUBLIC/'primary_plan.csv')
assert len(plan)==36 and plan.primary_execution_status.eq('not_executed').all()
assert plan.loc[plan.policy=='llm_with_fallback','planned_days'].sum()==252
display(plan.groupby('policy').agg(episodes=('matrix_index','size'),planned_policy_days=('planned_days','sum')))
auth_path=ROOT/'configs/runs/kaggle_primary.template.json'
authorization=json.loads(auth_path.read_text())
assert hashlib.sha256(auth_path.read_bytes().replace(b'\r\n',b'\n')).hexdigest()==qc['primary_authorization_template_sha256']
assert authorization['approved'] is False and authorization['budget_id'] is None
print({'primary_template_approved':authorization['approved'],'maximum_attempts_including_repairs':authorization['max_attempts'],'primary_episodes_executed':0})

,episodes,planned_policy_days
policy,,
adaptive,12,252
fixed,12,252
llm_with_fallback,12,252


{'primary_template_approved': False, 'maximum_attempts_including_repairs': 504, 'primary_episodes_executed': 0}


## Expense accounting and next stage

Stage 4 made no new model calls. The existing allowance retains all earlier known costs and full unknown-charge reservations. A primary run must explicitly extend task authorization under this same ledger; constructing a fresh budget to erase spending is rejected. The checked-in template grants no authority to spend.

Native results remain separate: 14 validated non-Retail tasks, the retained ICBE validity failure, and four deferred Retail tasks. This readiness checkpoint does not complete missing native coverage or establish real-world causal effects.

In [5]:
before=json.loads((PUBLIC/'preparation.json').read_text())['budget_snapshot']
assert qc['project_attempts_before_and_after']==before['total_project_attempts']
assert qc['inclusive_project_exposure_before_and_after_credits']==before['inclusive_project_exposure_credits']
display(pd.Series({k:before[k] for k in ['total_project_attempts','inclusive_project_exposure_credits','inclusive_cap_credits','remaining_kaggle_scope_credits','primary_base_projection_credits','primary_high_planning_projection_credits']}))
print('Forecasts are planning scenarios, not guaranteed bounds or independently verified provider billing.')
print('Next stage: explicitly authorized matched primary comparisons; no primary experiment is run by this notebook.')

total_project_attempts                      1438.000000
inclusive_project_exposure_credits             1.934083
inclusive_cap_credits                          6.000000
remaining_kaggle_scope_credits                 0.552549
primary_base_projection_credits                0.214936
primary_high_planning_projection_credits       0.429872
dtype: float64

Forecasts are planning scenarios, not guaranteed bounds or independently verified provider billing.
Next stage: explicitly authorized matched primary comparisons; no primary experiment is run by this notebook.
